# 第 13 章：機器學習基準模型、分類門檻與特徵比較

**學習目標**：
- 建立工作階段層級的購買預測資料。
- 區分數值與類別特徵，建立前處理 Pipeline。
- 使用 Logistic Regression 建立可重現的基準模型。
- 使用驗證資料依 F1 選擇分類門檻。
- 在保留測試集上評估 ROC AUC、F1 與分類報告。
- 加入 `session_month`，比較新特徵是否改善模型。

## 學習流程

1. 建立模型資料集
2. 檢查目標變數與類別不平衡
3. 定義基準特徵
4. 切分訓練、驗證與測試資料
5. 建立數值與類別前處理
6. 訓練 Logistic Regression
7. 使用驗證集調整分類門檻
8. 在測試集評估基準模型
9. 加入月份特徵並重新訓練
10. 比較模型與解讀結果

## 1. 環境設定與建立特徵資料

`build_features(data)` 會將工作階段、事件與客戶資料整理成一列一個工作階段的模型表，並建立：

- `page_view`：瀏覽事件數。
- `add_to_cart`：加入購物車事件數。
- `session_hour`：工作階段開始小時。
- `is_weekend`：是否為週末。
- `target`：是否出現購買事件，1 代表購買、0 代表未購買。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from common import build_features, ensure_packages, load_data

ensure_packages()
data = load_data()
model_df = build_features(data)

print(f"模型資料：{len(model_df):,} 列、{model_df.shape[1]} 欄")
display(model_df.head())

模型資料：70,000 列、15 欄


,session_id,customer_id,session_start,device,traffic_source,campaign,experiment_group,page_view,add_to_cart,purchase,segment,acquisition_channel,session_hour,is_weekend,target
0,1,2473,2025-08-14T11:02,desktop,sem,none,B,2,0,0,new,ads,11,0,0
1,2,2155,2024-02-29T11:56,mobile,sem,retarget,A,5,1,0,new,partner,11,0,0
2,3,2437,2024-11-13T16:23,mobile,sem,none,B,3,0,0,new,referral,16,0,0
3,4,164,2024-04-22T16:15,mobile,email,retarget,B,2,0,0,new,partner,16,0,0
4,5,1604,2024-06-15T08:39,desktop,direct,none,A,3,0,0,vip,ads,8,1,0


## 2. 檢查目標變數

分類模型開始前，應先確認正負類別數量。若正類比例很低，只看準確率可能產生誤導：模型全部預測為 0 也可能有很高準確率。

In [2]:
target_summary = model_df["target"].value_counts().sort_index().rename("筆數").to_frame()
target_summary["比例"] = target_summary["筆數"] / len(model_df)
target_summary.index = ["未購買（0）", "購買（1）"]
display(target_summary)

print(f"整體購買率：{model_df['target'].mean():.2%}")

,筆數,比例
未購買（0）,61347,0.876386
購買（1）,8653,0.123614


整體購買率：12.36%


## 3. 定義基準特徵

基準模型使用兩類特徵：

**數值特徵**：工作階段小時、週末旗標、瀏覽次數、加入購物車次數。

**類別特徵**：裝置、流量來源、活動、實驗組別、客群、客戶取得管道。

不使用 `purchase` 欄位，因為它直接定義 `target`，放入模型會造成目標洩漏。

In [3]:
base_numeric_features = [
    "session_hour",
    "is_weekend",
    "page_view",
    "add_to_cart",
]
base_categorical_features = [
    "device",
    "traffic_source",
    "campaign",
    "experiment_group",
    "segment",
    "acquisition_channel",
]
base_features = base_numeric_features + base_categorical_features

feature_types = pd.DataFrame({
    "特徵": base_features,
    "類型": ["數值"] * len(base_numeric_features)
    + ["類別"] * len(base_categorical_features),
})
display(feature_types)

,特徵,類型
0,session_hour,數值
1,is_weekend,數值
2,page_view,數值
3,add_to_cart,數值
4,device,類別
5,traffic_source,類別
6,campaign,類別
7,experiment_group,類別
8,segment,類別
9,acquisition_channel,類別


## 4. 為什麼需要訓練、驗證與測試三份資料？

- 訓練集：估計模型參數。
- 驗證集：選擇分類門檻或其他設定。
- 測試集：只在最後使用，估計模型對未見資料的表現。

原始程式在訓練資料上選擇門檻，雖未直接使用測試標籤，但可能得到較樂觀的門檻。本 Notebook 使用獨立驗證集，並讓所有模型共用相同切分，確保比較公平。

In [4]:
all_indices = model_df.index.to_numpy()

# 先保留 25% 作為最終測試集。
train_valid_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.25,
    random_state=42,
    stratify=model_df.loc[all_indices, "target"],
)

# 再從剩餘 75% 中取 20% 作驗證集，即全部資料的 15%。
train_idx, valid_idx = train_test_split(
    train_valid_idx,
    test_size=0.20,
    random_state=42,
    stratify=model_df.loc[train_valid_idx, "target"],
)

split_summary = pd.DataFrame({
    "資料集": ["訓練集", "驗證集", "測試集"],
    "筆數": [len(train_idx), len(valid_idx), len(test_idx)],
    "正類比例": [
        model_df.loc[train_idx, "target"].mean(),
        model_df.loc[valid_idx, "target"].mean(),
        model_df.loc[test_idx, "target"].mean(),
    ],
})
display(split_summary)

,資料集,筆數,正類比例
0,訓練集,42000,0.123619
1,驗證集,10500,0.123619
2,測試集,17500,0.123600


`stratify=y` 會讓各資料集的正類比例接近整體資料，降低隨機切分導致類別分布差異過大的風險。

## 5. 建立前處理與模型 Pipeline

數值與類別欄位需要不同處理：

- 數值：缺失值以中位數填補，再標準化。
- 類別：缺失值以眾數填補，再做 One-Hot Encoding。
- `handle_unknown="ignore"`：驗證或測試資料出現訓練時未見類別時，不會報錯。

所有步驟放入 Pipeline，可確保填補、縮放與編碼只在訓練集上學習，避免資料洩漏。

In [5]:
def build_model(numeric_features, categorical_features):
    """建立前處理與 Logistic Regression Pipeline。"""
    transformers = []

    if numeric_features:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),#如遇到空值，使用中位數填補
            ("scaler", StandardScaler()),#將數值各自計算 z-score，讓數值特徵的平均值為 0、標準差為 1
        ])
        transformers.append(("numeric", numeric_pipeline, numeric_features))

    if categorical_features:
        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),# 如遇到空值，使用眾數填補
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(("categorical", categorical_pipeline, categorical_features))

    preprocessor = ColumnTransformer(transformers=transformers)
    return Pipeline([
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(max_iter=400)),
    ])

## 6. 使用驗證集選擇最佳 F1 門檻

Logistic Regression 輸出購買機率。預設通常以 0.5 分類，但 0.5 不一定能得到最佳 F1。

Precision-Recall 曲線會提供不同門檻下的精確率與召回率，再計算：

`F1 = 2 × precision × recall ÷ (precision + recall)`

選擇驗證集 F1 最高的門檻，最後才用測試集評估。

In [6]:
def best_f1_threshold(y_true, probabilities):
    """依 Precision-Recall 曲線回傳 F1 最高的分類門檻。"""
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    if thresholds.size == 0:
        return 0.5

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / (precision[:-1] + recall[:-1] + 1e-12)
    )
    return float(thresholds[int(np.argmax(f1_values))])

## 7. 建立可重複使用的模型執行函式

函式會使用先前固定的資料切分：訓練模型、以驗證集選門檻，再於測試集計算 AUC、F1、精確率與召回率。

In [7]:
def run_model(df, numeric_features, categorical_features, model_name):
    """訓練模型、調整門檻並回傳測試評估結果。"""
    features = numeric_features + categorical_features
    X_train = df.loc[train_idx, features]
    y_train = df.loc[train_idx, "target"]
    X_valid = df.loc[valid_idx, features]
    y_valid = df.loc[valid_idx, "target"]
    X_test = df.loc[test_idx, features]
    y_test = df.loc[test_idx, "target"]

    model = build_model(numeric_features, categorical_features)
    model.fit(X_train, y_train)#計算

    valid_probability = model.predict_proba(X_valid)[:, 1]
    threshold = best_f1_threshold(y_valid, valid_probability)

    test_probability = model.predict_proba(X_test)[:, 1]
    prediction_default = (test_probability >= 0.5).astype(int)
    prediction_tuned = (test_probability >= threshold).astype(int)

    metrics = {
        "model": model_name,
        "auc": float(roc_auc_score(y_test, test_probability)),
        "threshold": threshold,
        "f1_default": float(f1_score(y_test, prediction_default, zero_division=0)),
        "f1_tuned": float(f1_score(y_test, prediction_tuned, zero_division=0)),
        "precision_tuned": float(precision_score(y_test, prediction_tuned, zero_division=0)),
        "recall_tuned": float(recall_score(y_test, prediction_tuned, zero_division=0)),
    }
    return {
        "model": model,
        "metrics": metrics,
        "y_test": y_test,
        "test_probability": test_probability,
        "prediction_default": prediction_default,
        "prediction_tuned": prediction_tuned,
    }

## 8. 訓練與評估基準模型

In [8]:
baseline_result = run_model(
    model_df,
    base_numeric_features,
    base_categorical_features,
    model_name="baseline",
)

display(pd.DataFrame([baseline_result["metrics"]]).round(4))

,model,auc,threshold,f1_default,f1_tuned,precision_tuned,recall_tuned
0,baseline,0.6894,0.1911,0.0227,0.3097,0.2722,0.3592


### AUC 與 F1 衡量不同面向

- ROC AUC：衡量模型排序正負類別的能力，不依賴單一分類門檻。
- F1：精確率與召回率的調和平均，依賴指定門檻。

因此調整門檻會改變 F1、precision 與 recall，但不會改變同一組預測機率的 AUC。

## 9. 基準模型分類報告

分類報告分別顯示類別 0 與 1 的 precision、recall、F1 及 support。對購買預測而言，應特別關注正類（1）的表現。

In [9]:
print("調整門檻後的分類報告：")
print(
    classification_report(
        baseline_result["y_test"],
        baseline_result["prediction_tuned"],
        digits=3,
        zero_division=0,
    )
)

baseline_confusion = confusion_matrix(
    baseline_result["y_test"],
    baseline_result["prediction_tuned"],
)
display(pd.DataFrame(
    baseline_confusion,
    index=["實際未購買", "實際購買"],
    columns=["預測未購買", "預測購買"],
))

調整門檻後的分類報告：
              precision    recall  f1-score   support

           0      0.905     0.865     0.885     15337
           1      0.272     0.359     0.310      2163

    accuracy                          0.802     17500
   macro avg      0.589     0.612     0.597     17500
weighted avg      0.827     0.802     0.813     17500



,預測未購買,預測購買
實際未購買,13260,2077
實際購買,1386,777


## 10. 新增工作階段月份特徵

融合 `practice_ch14.py`：將 `session_start` 轉成日期，再取月份 1～12。

月份在這裡當作數值特徵沿用原練習設計，但月份其實是循環資料：12 月與 1 月相鄰。若季節性很重要，可改用類別編碼，或建立正弦／餘弦循環特徵。

In [10]:
model_df_with_month = model_df.copy()
session_datetime = pd.to_datetime(
    model_df_with_month["session_start"], errors="coerce"
)
model_df_with_month["session_month"] = session_datetime.dt.month

print("session_month 缺失筆數：", model_df_with_month["session_month"].isna().sum())
display(model_df_with_month[["session_start", "session_month"]].head())

session_month 缺失筆數： 0


,session_start,session_month
0,2025-08-14T11:02,8
1,2024-02-29T11:56,2
2,2024-11-13T16:23,11
3,2024-04-22T16:15,4
4,2024-06-15T08:39,6


## 11. 訓練加入月份的模型

兩個模型使用完全相同的資料列切分，唯一差異是是否加入 `session_month`，因此效能差異較能反映新特徵的影響。

In [11]:
plus_month_numeric_features = base_numeric_features + ["session_month"]

month_result = run_model(
    model_df_with_month,
    plus_month_numeric_features,
    base_categorical_features,
    model_name="baseline + session_month",
)

display(pd.DataFrame([month_result["metrics"]]).round(4))

,model,auc,threshold,f1_default,f1_tuned,precision_tuned,recall_tuned
0,baseline + session_month,0.6894,0.1923,0.0227,0.3098,0.2734,0.3574


## 12. 比較兩個模型

除了觀察加入月份後的分數，也計算相對基準模型的變化量。小幅正負變動可能只是抽樣波動，不能只根據單次切分就宣稱特徵有效或無效。

In [12]:
model_comparison = pd.DataFrame([
    baseline_result["metrics"],
    month_result["metrics"],
])
display(model_comparison.round(4))

delta_auc = month_result["metrics"]["auc"] - baseline_result["metrics"]["auc"]
delta_f1 = month_result["metrics"]["f1_tuned"] - baseline_result["metrics"]["f1_tuned"]

print(f"AUC 變化：{delta_auc:+.4f}")
print(f"調整門檻後 F1 變化：{delta_f1:+.4f}")

,model,auc,threshold,f1_default,f1_tuned,precision_tuned,recall_tuned
0,baseline,0.6894,0.1911,0.0227,0.3097,0.2722,0.3592
1,baseline + session_month,0.6894,0.1923,0.0227,0.3098,0.2734,0.3574


AUC 變化：+0.0001
調整門檻後 F1 變化：+0.0001


## 13. 門檻選擇的商業意義

F1 將 precision 與 recall 視為同等重要，但實務成本可能不同：

- 假陽性成本高：例如對不會購買者發送昂貴優惠，應偏重 precision。
- 假陰性成本高：例如漏掉高潛力客戶，應偏重 recall。

因此最佳門檻不一定是 F1 最高點，而應結合預期收益、接觸成本與資源容量。

## 14. 練習題：將月份視為類別特徵

請把 `session_month` 從數值特徵移到類別特徵，重新訓練模型並比較 AUC 與 F1。

思考：數值月份假設 12 大於 1；類別月份則不建立大小順序。哪種表示方式更適合目前資料？

In [13]:
# TODO：可先遮住以下參考答案，再自行完成。
model_df_month_category = model_df_with_month.copy()
model_df_month_category["session_month"] = (
    model_df_month_category["session_month"].astype("Int64").astype("string")
)

month_as_category_result = run_model(
    model_df_month_category,
    base_numeric_features,
    base_categorical_features + ["session_month"],
    model_name="baseline + month as category",
)

exercise_comparison = pd.DataFrame([
    baseline_result["metrics"],
    month_result["metrics"],
    month_as_category_result["metrics"],
])
display(exercise_comparison.round(4))

,model,auc,threshold,f1_default,f1_tuned,precision_tuned,recall_tuned
0,baseline,0.6894,0.1911,0.0227,0.3097,0.2722,0.3592
1,baseline + session_month,0.6894,0.1923,0.0227,0.3098,0.2734,0.3574
2,baseline + month as category,0.6891,0.1712,0.0254,0.3211,0.2610,0.4170


## 常見錯誤與延伸

**常見錯誤**：
- 在切分資料前就對全部資料填補、縮放或編碼，造成資料洩漏。
- 使用測試資料反覆選特徵、參數或分類門檻，使測試分數失去客觀性。
- 將 `purchase` 特徵放入預測購買的模型，造成目標洩漏。
- 只看 accuracy，忽略類別不平衡及正類 recall。
- 認為預設 0.5 一定是最佳門檻。
- 兩個模型使用不同資料切分，導致效能比較不公平。

**延伸練習**：
- 使用交叉驗證比較特徵組合，降低單次切分的偶然性。
- 繪製 Precision-Recall 曲線與 ROC 曲線。
- 根據商業成本建立自訂門檻目標。
- 檢查 Logistic Regression 係數，但要注意 One-Hot 特徵的基準與正則化。
- 使用時間切分模擬模型預測未來資料的情境。

## 重點整理

- Pipeline 能把前處理與模型綁在一起，降低資料洩漏風險。
- 類別特徵需要編碼，數值特徵通常需要適當縮放。
- 門檻應在驗證資料上選擇，測試資料留到最後評估。
- AUC 衡量排序能力，F1 衡量指定門檻下 precision 與 recall 的平衡。
- 新特徵是否有價值，應在相同資料切分下比較，並進一步用交叉驗證確認。